Import libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Load dataset

In [3]:
df = pd.read_parquet("food_sample.parquet")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1000, 111)


,additives_n,additives_tags,allergens_tags,brands_tags,brands,categories,categories_tags,categories_properties,checkers_tags,ciqual_food_name_tags,...,stores_tags,stores,traces_tags,unique_scans_n,unknown_ingredients_n,unknown_nutrients_tags,vitamins_tags,with_non_nutritive_sweeteners,with_sweeteners,schema_version
0,0.0,[],[en:nuts],[xx:Bovetti],Bovetti,NaN,"[en:breakfasts, en:spreads, en:sweet-spreads, ...","{'ciqual_food_code': 31032.0, 'agribalyse_food...",[],[chocolate-spread-with-hazelnuts],...,[],NaN,"[en:milk, en:nuts, en:soybeans]",1.0,1.0,[],[],NaN,NaN,1004
1,0.0,[],[],[lagg-s],Lagg's,null,[en:null],"{'ciqual_food_code': None, 'agribalyse_food_co...",[],[unknown],...,None,NaN,[],1.0,0.0,[],[],NaN,NaN,999
2,0.0,[],[],[lagg-s],Lagg's,"Plant-based foods and beverages, Beverages, Ho...","[en:plant-based-foods-and-beverages, en:bevera...","{'ciqual_food_code': 18020.0, 'agribalyse_food...",[],[unknown],...,None,NaN,[],NaN,0.0,[],[],NaN,NaN,999
3,0.0,[],[],[xx:lagg-s],Lagg's,"Beverages and beverages preparations, Plant-ba...","[en:beverages-and-beverages-preparations, en:p...","{'ciqual_food_code': 18020.0, 'agribalyse_food...",[],[unknown],...,None,NaN,[],1.0,1.0,[],[],NaN,NaN,1002
4,0.0,[],[],[xx:Lagg's],Lagg's,NaN,[],"{'ciqual_food_code': None, 'agribalyse_food_co...",[],[unknown],...,[],NaN,[],NaN,0.0,[],[],NaN,NaN,1004


Check missing values

In [4]:
missing = df.isnull().sum().sort_values(ascending=False)

missing_pct = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing values": missing,
    "Missing (%)": missing_pct
}).sort_values("Missing (%)", ascending=False)

missing_summary.head(20)

,Missing values,Missing (%)
owner_fields,1000,100.0
owner,1000,100.0
photographers,999,99.9
new_additives_n,999,99.9
with_sweeteners,992,99.2
with_non_nutritive_sweeteners,974,97.4
editors,958,95.8
packagings_complete,926,92.6
environmental_score_score,893,89.3
manufacturing_places,880,88.0


Check dulicates


In [14]:
print("Code column exists:", "code" in df.columns)


Code column exists: True


In [15]:
print("Duplicated product codes:", df["code"].duplicated().sum())

Duplicated product codes: 0


Inspect useful columns

In [6]:
print(df.columns.tolist())

['additives_n', 'additives_tags', 'allergens_tags', 'brands_tags', 'brands', 'categories', 'categories_tags', 'categories_properties', 'checkers_tags', 'ciqual_food_name_tags', 'cities_tags', 'code', 'compared_to_category', 'complete', 'completeness', 'correctors_tags', 'countries_tags', 'created_t', 'creator', 'data_quality_errors_tags', 'data_quality_info_tags', 'data_quality_warnings_tags', 'data_sources_tags', 'environmental_score_data', 'environmental_score_grade', 'environmental_score_score', 'environmental_score_tags', 'editors', 'emb_codes_tags', 'emb_codes', 'entry_dates_tags', 'food_groups_tags', 'generic_name', 'images', 'informers_tags', 'ingredients_analysis_tags', 'ingredients_from_palm_oil_n', 'ingredients_n', 'ingredients_original_tags', 'ingredients_percent_analysis', 'ingredients_tags', 'ingredients_text', 'ingredients_with_specified_percent_n', 'ingredients_with_unspecified_percent_n', 'ingredients_without_ciqual_codes_n', 'ingredients_without_ciqual_codes', 'ingredi

In [17]:
# Select potentially useful columns

useful_columns = [
    "code",
    "product_name",
    "ingredients_text",
    "categories_tags",
    "allergens_tags",
    "quantity",
    "serving_size"
]

available_columns = [
    col for col in useful_columns
    if col in df.columns
]

print("Available columns:", available_columns)

Available columns: ['code', 'product_name', 'ingredients_text', 'categories_tags', 'allergens_tags', 'quantity', 'serving_size']


In [18]:
df_clean = df[available_columns].copy()

print("Original dataset:", df.shape)
print("Selected dataset:", df_clean.shape)

df_clean.head()

Original dataset: (1000, 111)
Selected dataset: (1000, 7)


,code,product_name,ingredients_text,categories_tags,allergens_tags,quantity,serving_size
0,0000101209159,"[{'lang': 'main', 'text': 'Véritable pâte à ta...","[{'lang': 'main', 'text': 'HAZELNUTS, 73% coco...","[en:breakfasts, en:spreads, en:sweet-spreads, ...",[en:nuts],350 g,NaN
1,0000105000011,"[{'lang': 'main', 'text': 'Chamomile Herbal Te...","[{'lang': 'main', 'text': 'CHAMOMILE FLOWERS.'...",[en:null],[],1 g,1g
2,0000105000042,"[{'lang': 'main', 'text': 'Lagg's, herbal tea,...","[{'lang': 'main', 'text': 'Peppermint.'}, {'la...","[en:plant-based-foods-and-beverages, en:bevera...",[],NaN,1.61 ONZ (45 g)
3,0000105000059,"[{'lang': 'main', 'text': 'Linden Flowers Tea'...","[{'lang': 'main', 'text': 'LINDEN FLOWERS.'}, ...","[en:beverages-and-beverages-preparations, en:p...",[],1.5 g,1.5g
4,0000105000073,"[{'lang': 'main', 'text': 'Herbal Tea, Hibiscu...","[{'lang': 'main', 'text': 'Hibiscus flowers.'}...",[],[],,1.5 g


In [19]:
missing_summary = pd.DataFrame({
    "Missing values": df_clean.isnull().sum(),
    "Missing (%)": (df_clean.isnull().mean() * 100).round(2)
})

missing_summary

,Missing values,Missing (%)
code,0,0.0
product_name,0,0.0
ingredients_text,0,0.0
categories_tags,94,9.4
allergens_tags,0,0.0
quantity,399,39.9
serving_size,152,15.2


Clean product names and ingredients

In [20]:
def extract_text(value):
    if isinstance(value, (list, np.ndarray)):
        texts = [
            item.get("text", "")
            for item in value
            if isinstance(item, dict)
        ]
        return " ".join(t for t in texts if isinstance(t, str) and t.strip())

    if isinstance(value, str):
        return value

    return np.nan


df_clean["product_name"] = df_clean["product_name"].apply(extract_text)

df_clean["ingredients_text"] = df_clean["ingredients_text"].apply(extract_text)

df_clean[["product_name", "ingredients_text"]].head()

,product_name,ingredients_text
0,Véritable pâte à tartiner noisettes chocolat n...,"HAZELNUTS, 73% cocoa dark chocolate (cocoa mas..."
1,Chamomile Herbal Tea Chamomile Herbal Tea,CHAMOMILE FLOWERS. CHAMOMILE FLOWERS. CHAMOMIL...
2,"Lagg's, herbal tea, peppermint Lagg's, herbal ...",Peppermint. Peppermint. Peppermint.
3,Linden Flowers Tea Linden Flowers Tea,LINDEN FLOWERS. LINDEN FLOWERS. LINDEN FLOWERS.
4,"Herbal Tea, Hibiscus Herbal Tea, Hibiscus",Hibiscus flowers. Hibiscus flowers. Hibiscus f...


In [21]:
print("Missing product names:",
      df_clean["product_name"].replace("", np.nan).isna().sum())

print("Missing ingredients:",
      df_clean["ingredients_text"].replace("", np.nan).isna().sum())

Missing product names: 14
Missing ingredients: 119


In [22]:
# Reload selected columns from original dataset
df_clean = df[available_columns].copy()

def extract_unique_text(value):
    if isinstance(value, (list, np.ndarray)):
        texts = [
            item.get("text", "").strip()
            for item in value
            if isinstance(item, dict)
            and isinstance(item.get("text"), str)
            and item.get("text", "").strip()
        ]

        # Keep the first non-empty text
        return texts[0] if texts else np.nan

    if isinstance(value, str):
        return value.strip() if value.strip() else np.nan

    return np.nan

df_clean["product_name"] = df_clean["product_name"].apply(extract_unique_text)
df_clean["ingredients_text"] = df_clean["ingredients_text"].apply(extract_unique_text)

df_clean[["product_name", "ingredients_text"]].head()

,product_name,ingredients_text
0,Véritable pâte à tartiner noisettes chocolat noir,"HAZELNUTS, 73% cocoa dark chocolate (cocoa mas..."
1,Chamomile Herbal Tea,CHAMOMILE FLOWERS.
2,"Lagg's, herbal tea, peppermint",Peppermint.
3,Linden Flowers Tea,LINDEN FLOWERS.
4,"Herbal Tea, Hibiscus",Hibiscus flowers.


delete products without relevant information

In [23]:
# Check rows before cleaning
print("Rows before:", len(df_clean))

# Remove products without names or ingredients
df_clean = df_clean.dropna(
    subset=["product_name", "ingredients_text"]
).copy()

print("Rows after:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Rows before: 1000
Rows after: 880
Rows removed: 120


In [24]:
print("Missing product names:",
      df_clean["product_name"].isna().sum())

print("Missing ingredients:",
      df_clean["ingredients_text"].isna().sum())

display(df_clean[["product_name", "ingredients_text"]].head(10))

Missing product names: 0
Missing ingredients: 0


,product_name,ingredients_text
0,Véritable pâte à tartiner noisettes chocolat noir,"HAZELNUTS, 73% cocoa dark chocolate (cocoa mas..."
1,Chamomile Herbal Tea,CHAMOMILE FLOWERS.
2,"Lagg's, herbal tea, peppermint",Peppermint.
3,Linden Flowers Tea,LINDEN FLOWERS.
4,"Herbal Tea, Hibiscus",Hibiscus flowers.
5,Apple & Cinnamon Tea,"TEA, CINNAMON & NATURAL APPLE FLAVOR."
6,Green Tea,GREEN TEA.
7,Shave Grass Herbal Tea,SHAVE GRASS.
8,"Lagg's, herbal tea, chamomile * mint",Chamomile spearmint.
9,Artichoke Herbal Tea,ARTICHOKE MALVA SENNA LEAF HIBISCUS CHAMOMILE ...


In [25]:
df_clean.head(10)

,code,product_name,ingredients_text,categories_tags,allergens_tags,quantity,serving_size
0,0000101209159,Véritable pâte à tartiner noisettes chocolat noir,"HAZELNUTS, 73% cocoa dark chocolate (cocoa mas...","[en:breakfasts, en:spreads, en:sweet-spreads, ...",[en:nuts],350 g,NaN
1,0000105000011,Chamomile Herbal Tea,CHAMOMILE FLOWERS.,[en:null],[],1 g,1g
2,0000105000042,"Lagg's, herbal tea, peppermint",Peppermint.,"[en:plant-based-foods-and-beverages, en:bevera...",[],NaN,1.61 ONZ (45 g)
3,0000105000059,Linden Flowers Tea,LINDEN FLOWERS.,"[en:beverages-and-beverages-preparations, en:p...",[],1.5 g,1.5g
4,0000105000073,"Herbal Tea, Hibiscus",Hibiscus flowers.,[],[],,1.5 g
5,0000105000196,Apple & Cinnamon Tea,"TEA, CINNAMON & NATURAL APPLE FLAVOR.",[en:null],[],1.5 g,1.5g
6,0000105000219,Green Tea,GREEN TEA.,[en:null],[],30 g,30.05047g
7,0000105000318,Shave Grass Herbal Tea,SHAVE GRASS.,[en:null],[],30 g,30.05047g
8,0000105000356,"Lagg's, herbal tea, chamomile * mint",Chamomile spearmint.,"[en:plant-based-foods-and-beverages, en:bevera...",[],NaN,1.06 ONZ (30 g)
9,0000105000363,Artichoke Herbal Tea,ARTICHOKE MALVA SENNA LEAF HIBISCUS CHAMOMILE ...,[en:null],[],30 g,30.05047g


Clean categories and allergens

In [26]:
def clean_tags(value):
    if isinstance(value, (list, np.ndarray)):
        return [
            tag.replace("en:", "").strip()
            for tag in value
            if isinstance(tag, str)
            and tag.strip().lower() not in ("", "en:null")
        ]
    return []

df_clean["categories_tags"] = df_clean["categories_tags"].apply(clean_tags)

df_clean[["product_name", "categories_tags"]].head(10)

,product_name,categories_tags
0,Véritable pâte à tartiner noisettes chocolat noir,"[breakfasts, spreads, sweet-spreads, fr:pates-..."
1,Chamomile Herbal Tea,[]
2,"Lagg's, herbal tea, peppermint","[plant-based-foods-and-beverages, beverages, h..."
3,Linden Flowers Tea,"[beverages-and-beverages-preparations, plant-b..."
4,"Herbal Tea, Hibiscus",[]
5,Apple & Cinnamon Tea,[]
6,Green Tea,[]
7,Shave Grass Herbal Tea,[]
8,"Lagg's, herbal tea, chamomile * mint","[plant-based-foods-and-beverages, beverages, h..."
9,Artichoke Herbal Tea,[]


Noramlize allergens

In [27]:
df_clean["allergens_tags"] = df_clean["allergens_tags"].apply(clean_tags)

df_clean[["product_name", "allergens_tags"]].head(10)

,product_name,allergens_tags
0,Véritable pâte à tartiner noisettes chocolat noir,[nuts]
1,Chamomile Herbal Tea,[]
2,"Lagg's, herbal tea, peppermint",[]
3,Linden Flowers Tea,[]
4,"Herbal Tea, Hibiscus",[]
5,Apple & Cinnamon Tea,[]
6,Green Tea,[]
7,Shave Grass Herbal Tea,[]
8,"Lagg's, herbal tea, chamomile * mint",[]
9,Artichoke Herbal Tea,[]


Check results

In [28]:
print("Products:", len(df_clean))

print("Products without categories:",
      df_clean["categories_tags"].apply(len).eq(0).sum())

print("Products without allergen information:",
      df_clean["allergens_tags"].apply(len).eq(0).sum())

display(
    df_clean[
        ["product_name", "categories_tags", "allergens_tags"]
    ].head(10)
)

Products: 880
Products without categories: 457
Products without allergen information: 324


,product_name,categories_tags,allergens_tags
0,Véritable pâte à tartiner noisettes chocolat noir,"[breakfasts, spreads, sweet-spreads, fr:pates-...",[nuts]
1,Chamomile Herbal Tea,[],[]
2,"Lagg's, herbal tea, peppermint","[plant-based-foods-and-beverages, beverages, h...",[]
3,Linden Flowers Tea,"[beverages-and-beverages-preparations, plant-b...",[]
4,"Herbal Tea, Hibiscus",[],[]
5,Apple & Cinnamon Tea,[],[]
6,Green Tea,[],[]
7,Shave Grass Herbal Tea,[],[]
8,"Lagg's, herbal tea, chamomile * mint","[plant-based-foods-and-beverages, beverages, h...",[]
9,Artichoke Herbal Tea,[],[]


## Standardize quantities and serving sizes

In [29]:
import re

def convert_to_grams(value):
    if pd.isna(value):
        return np.nan

    text = str(value).lower().strip()

    # Prefer the gram value inside parentheses
    match = re.search(r'\(([\d.,]+)\s*g\)', text)
    if match:
        return float(match.group(1).replace(',', '.'))

    # Convert kilograms to grams
    match = re.fullmatch(r'([\d.,]+)\s*kg', text)
    if match:
        return float(match.group(1).replace(',', '.')) * 1000

    # Extract quantities already expressed in grams
    match = re.fullmatch(r'([\d.,]+)\s*g', text)
    if match:
        return float(match.group(1).replace(',', '.'))

    return np.nan

Apply it to our dataframe

In [30]:
df_clean["quantity_g"] = df_clean["quantity"].apply(convert_to_grams)

df_clean["serving_size_g"] = df_clean["serving_size"].apply(convert_to_grams)

display(
    df_clean[
        ["product_name", "quantity", "quantity_g",
         "serving_size", "serving_size_g"]
    ].head(15)
)

,product_name,quantity,quantity_g,serving_size,serving_size_g
0,Véritable pâte à tartiner noisettes chocolat noir,350 g,350.0,NaN,NaN
1,Chamomile Herbal Tea,1 g,1.0,1g,1.00000
2,"Lagg's, herbal tea, peppermint",NaN,NaN,1.61 ONZ (45 g),45.00000
3,Linden Flowers Tea,1.5 g,1.5,1.5g,1.50000
4,"Herbal Tea, Hibiscus",,NaN,1.5 g,1.50000
5,Apple & Cinnamon Tea,1.5 g,1.5,1.5g,1.50000
6,Green Tea,30 g,30.0,30.05047g,30.05047
7,Shave Grass Herbal Tea,30 g,30.0,30.05047g,30.05047
8,"Lagg's, herbal tea, chamomile * mint",NaN,NaN,1.06 ONZ (30 g),30.00000
9,Artichoke Herbal Tea,30 g,30.0,30.05047g,30.05047


Check how many values we have converted

In [31]:
print("Products:", len(df_clean))

print("Quantity converted:",
      df_clean["quantity_g"].notna().sum())

print("Serving size converted:",
      df_clean["serving_size_g"].notna().sum())

print("Missing quantity in grams:",
      df_clean["quantity_g"].isna().sum())

print("Missing serving size in grams:",
      df_clean["serving_size_g"].isna().sum())

Products: 880
Quantity converted: 423
Serving size converted: 755
Missing quantity in grams: 457
Missing serving size in grams: 125


## Final data validation

In [32]:
# Check unusual serving sizes

print(df_clean["serving_size_g"].describe())

print("Zero or negative serving sizes:",
      (df_clean["serving_size_g"] <= 0).sum())

display(
    df_clean[
        df_clean["serving_size_g"] > 1000
    ][["product_name", "serving_size", "serving_size_g"]].head(10)
)

count     755.000000
mean       43.282746
std       112.919550
min         0.490000
25%        28.000000
50%        36.000000
75%        40.000000
max      2940.000000
Name: serving_size_g, dtype: float64
Zero or negative serving sizes: 0


,product_name,serving_size,serving_size_g
873,Freshmints,60 serving (2940 g),2940.0


In [34]:
# Flag suspicious serving sizes

df_clean["serving_size_review"] = (
    df_clean["serving_size_g"] > 1000
)

# Keep original data, but exclude suspicious values
df_clean.loc[
    df_clean["serving_size_review"],
    "serving_size_g"
] = np.nan

print("Products requiring review:",
      df_clean["serving_size_review"].sum())

Products requiring review: 1


Safe the clean dataset

In [35]:
# Export cleaned dataset

df_clean.to_parquet(
    "food_clean.parquet",
    index=False
)

print("Clean dataset saved successfully!")
print("Final shape:", df_clean.shape)

Clean dataset saved successfully!
Final shape: (880, 10)
